In [1]:
#STEP 0: Load dataset
import pandas as pd

df = pd.read_csv("Reviews.csv")

# Use only needed column
df = df[["Text", "Score"]].dropna()

df.rename(columns={"Text": "review_text"}, inplace=True)

df.head()

,review_text,Score
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1
2,This is a confection that has been around a fe...,4
3,If you are looking for the secret ingredient i...,2
4,Great taffy at a great price. There was a wid...,5


In [5]:
#TASK 1: Preprocessing
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') 

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()  # lowercase
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # tokenization
    tokens = word_tokenize(text)
    
    # remove stopwords
    tokens = [w for w in tokens if w not in stop_words]
    
    # lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    
    return " ".join(tokens)

df["clean_text"] = df["review_text"].apply(preprocess)

df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\R0Y\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\R0Y\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\R0Y\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\R0Y\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


,review_text,Score,clean_text
0,I have bought several of the Vitality canned d...,5,bought several vitality canned dog food produc...
1,Product arrived labeled as Jumbo Salted Peanut...,1,product arrived labeled jumbo salted peanutsth...
2,This is a confection that has been around a fe...,4,confection around century light pillowy citrus...
3,If you are looking for the secret ingredient i...,2,looking secret ingredient robitussin believe f...
4,Great taffy at a great price. There was a wid...,5,great taffy great price wide assortment yummy ...


In [6]:
#TASK 2: Vocabulary Creation
from collections import Counter

all_words = " ".join(df["clean_text"]).split()

vocab = Counter(all_words)

print("Vocabulary Size:", len(vocab))

# Top frequent words
print(vocab.most_common(20))

Vocabulary Size: 231812
[('br', 264693), ('like', 263536), ('taste', 211889), ('good', 197055), ('one', 189267), ('flavor', 180518), ('product', 176805), ('coffee', 170078), ('great', 163577), ('love', 163056), ('tea', 148838), ('food', 147840), ('would', 123364), ('get', 118935), ('make', 107129), ('dog', 105866), ('really', 100417), ('time', 97346), ('dont', 95556), ('much', 91906)]


In [7]:
#TASK 3: Feature Engineering
#1. One Hot Encoding (manual)
unique_words = list(set(all_words))

def one_hot(doc):
    return [1 if word in doc.split() else 0 for word in unique_words]

ohe_matrix = [one_hot(doc) for doc in df["clean_text"][:100]]  # limit for memory

# Bag of Words

from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df["clean_text"])

print(X_bow.shape)

(568454, 5000)


In [8]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df["clean_text"])

print(X_tfidf.shape)

(568454, 5000)


In [9]:
#TASK 4: Comparison Analysis
import pandas as pd

comparison = pd.DataFrame({
    "Method": ["OHE", "BoW", "TF-IDF"],
    "Captures Frequency": ["No", "Yes", "Yes"],
    "Captures Importance": ["No", "No", "Yes"],
    "Sparse": ["High", "High", "High"],
    "Semantic Understanding": ["No", "No", "No"]
})

comparison

,Method,Captures Frequency,Captures Importance,Sparse,Semantic Understanding
0,OHE,No,No,High,No
1,BoW,Yes,No,High,No
2,TF-IDF,Yes,Yes,High,No


In [10]:
# TASK 5: Sparse Matrix Analysis
import numpy as np

def sparsity(matrix):
    return 1.0 - (matrix.count_nonzero() / (matrix.shape[0] * matrix.shape[1]))

print("BoW Sparsity:", sparsity(X_bow))
print("TF-IDF Sparsity:", sparsity(X_tfidf))

BoW Sparsity: 0.9938053495973289
TF-IDF Sparsity: 0.9938053495973289


In [11]:
# TASK 7: Sentiment Classification
# Convert labels
df["sentiment"] = df["Score"].apply(lambda x: 1 if x >= 4 else 0)
# Train model (BoW)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_bow, df["sentiment"], test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("BoW Accuracy:", accuracy_score(y_test, pred))

#Train model (TF-IDF)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, df["sentiment"], test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("TF-IDF Accuracy:", accuracy_score(y_test, pred))

f:\CodeZ\AI\PythonWebScrapper\scrapper1\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


BoW Accuracy: 0.8886631307667273
TF-IDF Accuracy: 0.8901496160645962


In [12]:
# Improvements
feature_names = tfidf.get_feature_names_out()

top_words = sorted(
    zip(model.coef_[0], feature_names),
    reverse=True
)[:10]

print("Top Positive Words:", top_words)

Top Positive Words: [(np.float64(10.698840499003053), 'great'), (np.float64(9.221022571061148), 'perfect'), (np.float64(9.044578694454666), 'delicious'), (np.float64(8.966948504212743), 'highly'), (np.float64(8.874038654484917), 'best'), (np.float64(8.478587421080931), 'love'), (np.float64(8.139913492019208), 'excellent'), (np.float64(7.521382966923247), 'amazing'), (np.float64(7.493262568222204), 'hooked'), (np.float64(7.025855868301798), 'wonderful')]
